→「ランタイムのタイプを変更」→ T4 GPU

In [ ]:
# 1. GPU (T4等) が利用可能か自動チェック
import sys
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPUが有効です: {gpu_name}")
else:
    # CPUの場合は警告を出す
    print("⚠️ 警告: GPUが有効になっていません！")
    print("Colabの上部メニュー [ランタイム] -> [ランタイムのタイプを変更] から『T4 GPU』を選択してください。")

OllamaのAPI：AIに「JSONを送って、JSONを受け取る」会話する

Ollamaをインストール  （初回のみ 1回だけ）

WARNINGは無視する

In [ ]:
# Ollamaをインストール（zstd・pciutilsはインストールに必要な補助ツール）
!apt-get -qq install -y zstd pciutils > /dev/null   # Linux コマンド
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Ollamaサーバーをバックグラウンドで起動
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(7)  # 起動を待つ

Qwen3.5-4Bをダウンロード

In [ ]:
# Qwen3.5-4Bをダウンロード（q8_0 約4〜5GB版を選んでいる）

MODEL = "qwen3.5:4b-q8_0"
!ollama pull {MODEL}

print("ダウンロードが終わりました。　次に進めます")

最初のテスト

Ollamaを通して、Colabの中で動いている「AIサーバー」にプロンプトを送る。
 http://localhost:11434。ここに**JSON書式のデータを送ると、JSONで答えが返ってくる。
 数回、繰り返して、答えが違ってくるのを確認しましょう。　なせ、答えが毎回違うのか？

In [ ]:
import requests, json

URL = "http://localhost:11434/api/chat"  # ポート番号

# 送るデータ（リクエスト）
data = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "こんにちは。好きな事を教えて"}
    ],
    "stream": False,   # 答えを一度にまとめて受け取る
    "think": False,     # 考える過程は出さない
    "options": {"num_predict": 200},   # 出力は最大150トークンまで
}

response = requests.post(URL, json=data)   # 送信
answer = response.json()                    # 返ってきたJSON

"""
print("===== 送ったJSON =====", "\n")
print(json.dumps(data, ensure_ascii=False, indent=2))
print("\n===== 返ってきたJSON =====")
print(json.dumps(answer, ensure_ascii=False, indent=2))
"""

# 答えの部分だけを取り出す
print("\n===== 【AIの答えの部分】 =====")
print(answer["message"]["content"])

In [ ]:
# -------------------------------------
# 【関数 ask_letter】回答を受け取る関数
# -------------------------------------

def ask_letter(messages):
    data = {"model": MODEL, "messages": messages, "stream": False, "think": False}
    res = requests.post(URL, json=data).json()
    print(f"（AIが読んだ量：{res['prompt_eval_count']} トークン）")
    return res["message"]["content"]

system：キャラクターを定義する

system；キャラクター・口調・ルールの設定。AIにやらせる「台本」

In [ ]:
# キャラクターを決める

question = "おすすめの野菜を1つ教えてよー"

messages = [
    {"role": "system", "content": "あなたは関西弁で話す八百屋の元気な店主です。100程度で答えます。"},
    {"role": "user",   "content": question}
]
print(ask_letter(messages))

"system"の内容を変えて、もう一度実行する

In [ ]:
# ★ system の内容を書き換えて、もう一度実行する
messages = [
    {"role": "system", "content": "あなたは小学生に教える女性の先生です。やさしく短く答えます。"},
    {"role": "user",   "content": question}
]
print(ask_letter(messages))

In [ ]:
# ★ system の内容を書き換えて、もう一度実行する
messages = [
    {"role": "system", "content": "あなたはNHK「今日の料理」に出演する有名な料理人です。専門的に答えて"},
    {"role": "user",   "content": question}
]
print(ask_letter(messages))

======  プロンプトの対話履歴を残さない場合  ===========

履歴を残さずに同じ会話を２回する

In [ ]:
# 1回目：自分の名前を教える

print(ask_letter([{"role": "user", "content": "私の名前は茅野愛衣です。覚えておいてね。"}]))

In [ ]:
# 2回目：名前を聞く（1回目のやりとりは送っていない）

print(ask_letter([{"role": "user", "content": "私の名前は何ですか？"}]))

AIは名前を答えられない！。AIは会話を学習しないし、覚えてない。

1回目と2回目は、AIにとってまったく別の、初対面の会話になっている。

======  プロンプトの対話履歴を残す場合  ===========

（プロンプト＋回答）を残すことで、会話が進んでいく

In [ ]:
# 関数【Chat_history】

history = [
    {"role": "system", "content": "あなたは親切なアシスタントです。短く答えます。"}
]

# -----------------------------
# historyに次々に足し込んでいく
# -----------------------------
def chat_history(user_input):
    history.append({"role": "user", "content": user_input})        # ① 自分の発言を足す
    reply = ask_letter(history)                                    # ② 履歴をまるごと送る
    history.append({"role": "assistant", "content": reply})        # ③ AIの答えも足す
    return reply

In [ ]:
user_input = "私の名前は小芝風花です。覚えておいてね。"

print("あなた⇒：", user_input)
print("AI　⇒　：", chat_history(user_input))

# 今、AIに毎回送った履歴
for m in history:
    print(f"[{m['role']}] {m['content']}")

履歴のリセット　白紙に戻す

In [ ]:
# 履歴をリセットする

history = [
    {"role": "system", "content": "あなたは親しみやすいアシスタントです。短く答えます。"}
]

"""
# -----------------------------
# historyに次々に足し込んでいく
# -----------------------------
def chat_history(user_input):
    history.append({"role": "user", "content": user_input})        # ① 自分の発言を足す
    reply = ask_letter(history)                                    # ② 履歴をまるごと送る
    history.append({"role": "assistant", "content": reply})        # ③ AIの答えも足す
    return reply
"""
print("履歴をリセットしました")

In [ ]:
# 名前を再度聞く
print(ask_letter([{"role": "user", "content": "私の名前は何ですか？"}]))